In [ ]:
import json
import pandas as pd
import rasterio
import os
import numpy as np
import matplotlib.pyplot as plt

from scipy.signal import convolve2d

In [ ]:
rundir = "/home/ebr/projects/release-volume-sampler/generated/messina_001"
volumes_path = os.path.join(rundir,"volumes/recursive_propagations.json")
# Load the JSON file
with open(volumes_path, "r") as f:
    data = json.load(f)

triangulation_dir = os.path.join(rundir, "triangulation")
tri_mask_path = os.path.join(triangulation_dir, "triangulation.tif")

In [ ]:
# Normalize the "volumes" data within each entry
df = pd.json_normalize(
    data,
    record_path=["volumes"],  # Extract and flatten the "volumes" list
    meta=["seed_triangle", "seed_triangle_probability"]  # Include these fields as metadata
)

# Keep 'released' as a list and skip expanding 'steps'
# Drop 'steps' if it's not needed
df = df.drop(columns=["steps"])
df['id'] = df.index

In [ ]:
df["probability"] = df.seed_triangle_probability*df.condprob # Use seed probability to set final probability

In [ ]:
df.shape

Statistical relation for assigning depth of the volume (Zengafinnen-Morris et al. 2022): $V = 0.0298*A^{1.36}$


In [ ]:
df["volume"] = 0.0298*df.area**1.36
df["thickness"] = df.volume/df.area

In [ ]:
df.area.plot(kind="hist", weights=df.probability, bins=30, title="Weighted area of release")

In [ ]:
df.thickness.plot(kind="hist", weights=df.probability, bins=30, title="Weighted thickness of release")

In [ ]:
df.drop(columns=["released"]).to_csv(os.path.join(rundir, "volumes/volumes.csv"))

In [ ]:
# write volumes to rasters
df_filtered = df.loc[(df.max_elevation > -500) & (df.area > 1300000) & (df.probability > 1e-6)]
df_filtered.sort_values("area", ascending=False)


In [ ]:
df_filtered.loc[df_filtered.seed_triangle == 1998]

In [ ]:
with rasterio.open(tri_mask_path) as src:
    tri_mask = src.read(1)  # Read the triangle mask
    tri_profile = src.profile  # Copy metadata to use in output

with rasterio.open("/home/ebr/projects/release-volume-sampler/generated/messina_001/bathy_truncated.tif") as src:
    bathy = src.read(1)  # Read the triangle mask
    bathy_profile = src.profile  # Copy metadata to use in output

# Update profile for single-band, unsigned 8-bit data
bathy_profile.update(dtype=rasterio.float32, count=1)

#added_volumes = np.zeros((profile["height"], profile["width"]))

for i, volume in df_filtered.iterrows():
    print(f"i: {i}, volume:{volume}")
    
    # Create binary volume mask: 1 if pixel belongs to specified triangles, else 0
    volume_mask = np.isin(tri_mask, volume.released)
    

    # Smooth volume
    volume_raster = convolve2d(volume_mask.astype(float)*volume.thickness, np.ones((3,3))/9., mode="same")
    
    volume_path = os.path.join(rundir, f"volumes/rasters/volume_seed-{volume.seed_triangle}_area-{volume.area:.2e}_id-{volume.id}.tif")

    with rasterio.open(volume_path, 'w', **bathy_profile) as dst:
        dst.write(volume_raster.astype(rasterio.float32), 1)  # Write volume to file

In [ ]:
profile

In [ ]:
bathy_profile

In [ ]:
plt.imshow(volume_raster_c)

In [ ]:
volume_raster[volume_raster == 0.] = np.nan


## Find slopeunits

In [ ]:
import os
import numpy as np


In [ ]:
os.chdir("/home/ebr/projects/release-volume-sampler")

In [ ]:
from src.volume_sampler.release_volume_sampler import RecursiveReleaseAnalysis

# Usage example
config = {
    "rundir": "/home/ebr/projects/release-volume-sampler/generated/messina_001",
    "mesh_path": "/home/ebr/projects/release-volume-sampler/generated/messina_001/triangulation/triangulation.vtk",
    "cumprob_logfos_path": "/home/ebr/projects/release-volume-sampler/generated/messina_001/triangulation/cummulative_fos.npz",
    "utm_epsg_code": 32633, # Messina strait
}

run_config = {
    "fos_threshold": 1.1,
    "recursive_probability_threshold": 0.01,
    "seed_triangle_probability_threshold": 0.1,
}
# Execute analysis.
analysis = RecursiveReleaseAnalysis(**config)

In [ ]:
r2 = (analysis.normals**2).sum(axis=1)
slope = np.rad2deg(np.arccos(1/np.sqrt(r2)))

In [ ]:
plt.hist(analysis.slopes, bins=40)

In [ ]:
analysis.slopes[51]

In [ ]:
slopeunits = np.zeros(analysis.n_triangles, dtype=int)
triangles = list(range(analysis.n_triangles))
slopeunit = 1
while(triangles):
    print(len(triangles))
    triangle = triangles.pop()
    next_upstream = analysis.get_upstream_triangles(triangle).tolist()
    
    upstream_triangles = []
    while(next_upstream):
        previous_upstream = next_upstream.copy()
        next_upstream = []
        for t in previous_upstream:
            next_upstream.extend(analysis.get_upstream_triangles(t).tolist())
        upstream_triangles.extend(next_upstream)
    
    # Remove duplicates
    upstream_triangles = list(set(upstream_triangles))
    if len(upstream_triangles) > 0:
        upstream_slopeunits = slopeunits[np.array(upstream_triangles)]
        if upstream_slopeunits.max() == 0:
            slopeunits[triangle] = slopeunit
            slopeunits[upstream_triangles] = slopeunit
            [triangles.remove(t) for t in upstream_triangles]
        else:
            slopeunits[triangle] = upstream_slopeunits.max()
    else:
        slopeunit += 1
        slopeunits[triangle] = slopeunit
    
    #print(f"triangle: {triangle}, upstream: {upstream_slopeunits.max()}")